In [1]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [2]:
MOVIES_PATH  = "movies.csv"   # adjust path if needed
RATINGS_PATH = "ratings.csv"  # optional — place alongside movies.csv

if not os.path.exists(MOVIES_PATH):
    raise FileNotFoundError(
        f"'{MOVIES_PATH}' not found. "
        "Make sure it is in the same folder as this script."
    )

movies = pd.read_csv(MOVIES_PATH)
print(f"      Loaded {len(movies):,} movies  |  columns: {movies.columns.tolist()}")


      Loaded 62,423 movies  |  columns: ['movieId', 'title', 'genres']


In [3]:

movies.dropna(subset=["title", "genres"], inplace=True)

movies = movies[movies["genres"] != "(no genres listed)"].copy()
movies.reset_index(drop=True, inplace=True)

movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")


movies["title_clean"] = (
    movies["title"]
    .str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
    .str.strip()
)

print(f"      After cleaning: {len(movies):,} movies remain.")


      After cleaning: 57,361 movies remain.


In [4]:
if os.path.exists(RATINGS_PATH):
    ratings = pd.read_csv(RATINGS_PATH)
    print(f"      ratings.csv loaded — {len(ratings):,} rows.")

    avg_ratings = (
        ratings.groupby("movieId")["rating"]
        .agg(avg_rating="mean", vote_count="count")
        .reset_index()
    )

    C = avg_ratings["avg_rating"].mean()   # global mean
    m = avg_ratings["vote_count"].quantile(0.25)  # minimum votes threshold

    avg_ratings["bayesian_rating"] = (
        (avg_ratings["vote_count"] * avg_ratings["avg_rating"] + m * C)
        / (avg_ratings["vote_count"] + m)
    )

    movies = movies.merge(avg_ratings[["movieId", "bayesian_rating"]], on="movieId", how="left")
    movies["bayesian_rating"].fillna(C, inplace=True)  # fill missing with global mean

    print(f"      Ratings merged. Global mean: {C:.2f}")
    USE_RATINGS = True
else:
    print("      ratings.csv not found — skipping ratings boost (purely content-based).")
    movies["bayesian_rating"] = 5.0  # neutral placeholder
    USE_RATINGS = False


      ratings.csv loaded — 25,000,095 rows.
      Ratings merged. Global mean: 3.07


In [5]:
movies["genres_clean"] = (
    movies["genres"]
    .str.replace("|", " ", regex=False)
    .str.replace("-", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace("&", "and", regex=False)
    .str.lower()
)

# Combine genres + clean title into one tag string
movies["tags"] = movies["genres_clean"] + " " + movies["title_clean"].str.lower()

In [6]:
MAX_MOVIES = 10_000
if len(movies) > MAX_MOVIES:
    print(f"      Dataset has {len(movies):,} movies — using top {MAX_MOVIES:,} for performance.")
    movies = movies.head(MAX_MOVIES).copy()
    movies.reset_index(drop=True, inplace=True)

cv = CountVectorizer(max_features=5_000, stop_words="english")
vectors = cv.fit_transform(movies["tags"]).toarray()

similarity_raw = cosine_similarity(vectors)

if USE_RATINGS:
    
    max_r = movies["bayesian_rating"].max()
    rating_norm = (movies["bayesian_rating"] / max_r).values  # shape (N,)
    similarity = similarity_raw * rating_norm[np.newaxis, :]  # broadcast
    print("      Ratings boost applied.")
else:
    similarity = similarity_raw

print(f"      Similarity matrix shape: {similarity.shape}")


      Dataset has 57,361 movies — using top 10,000 for performance.


      Ratings boost applied.
      Similarity matrix shape: (10000, 10000)


In [7]:
movies_to_save = movies[["movieId", "title", "title_clean", "genres"]].copy()

with open("movies_list.pkl", "wb") as f:
    pickle.dump(movies_to_save, f)

with open("similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

print()
print(" Done!")


 Done!
